# UTS Big Data & Predictive Analytics Lanjut (ST153)
## Notebook 1 (PC Master): Ingestion Data Stream ke Apache Kafka & Ekspor Snapshot

**Konfigurasi Klaster:**
- **PC Master / Kafka Broker:** `192.168.110.109:9092`
- **Topic:** `wikipedia-recentchange` (4 Partisi)
- **Shared Folder:** `Z:/` (Pastikan tombol *"Petakan share Z:"* di Panel Lab sudah diklik)

**Petunjuk Eksekusi:**
1. Jalankan **Cell 1** untuk memastikan pustaka `confluent-kafka` dan `requests` terpasang.
2. Jalankan **Cell 2** untuk membuat *topic* `wikipedia-recentchange` (4 partisi).
3. Jalankan **Cell 3** (*Producer*) dan biarkan menyala minimal **10 menit** (hingga terkumpul ±200.000 pesan atau sesuai target).
4. Saat *Producer* berjalan, buka tab notebook baru atau jalankan **Cell 4 & Cell 5** setelah *Producer* dihentikan sementara untuk mengambil bukti Soal 1b, Soal 1c, dan mengekspor snapshot ke `Z:/wiki_snapshot.jsonl`.

In [1]:
# CELL 1: Instalasi Pustaka Klien Kafka & HTTP Stream (Kompatibel Python 3.14)
!pip install --upgrade confluent-kafka requests


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# CELL 2: Pembuatan Topic Kafka 'wikipedia-recentchange' (Soal 1a)
from confluent_kafka.admin import AdminClient, NewTopic

KAFKA_BROKER = '192.168.110.109:9092'
TOPIC_NAME = 'wikipedia-recentchange'

admin = AdminClient({'bootstrap.servers': KAFKA_BROKER})

topic_baru = NewTopic(
    topic=TOPIC_NAME,
    num_partitions=4,
    replication_factor=1,
    config={"retention.ms": "86400000"}
)

futures = admin.create_topics([topic_baru])

for topic, future in futures.items():
    try:
        future.result()
        print(f"[SUKSES] Topic '{topic}' dengan 4 partisi berhasil dibuat!")
    except Exception as e:
        print(f"[INFO] Status topic '{topic}': {e}")

[INFO] Status topic 'wikipedia-recentchange': KafkaError{code=_TIMED_OUT,val=-185,str="Failed while waiting for controller: Local: Timed out"}


In [ ]:
# CELL 3: Producer Python SSE Wikimedia -> Kafka (Soal 1b)
# Biarkan cell ini berjalan minimal 10 menit. Tekan tombol Stop (■) jika sudah cukup.
import json
import time
import hashlib
import ipaddress
import requests
from confluent_kafka import Producer

# Konfigurasi Utama
KAFKA_BROKER = '192.168.110.109:9092'
TOPIC_NAME = 'wikipedia-recentchange'
STREAM_URL = 'https://stream.wikimedia.org/v2/stream/recentchange'

# User-Agent disesuaikan dengan permintaan soal UTS
HEADERS = {
    'User-Agent': 'AmikomBigDataUTS/1.0 (https://amikom.ac.id; mahasiswa@students.amikom.ac.id)'
}

def anonymize_user(username):
    """Menyamarkan alamat IP pengguna anonim dengan hash SHA-256."""
    if not username:
        return "unknown"
    try:
        ipaddress.ip_address(username)
        hashed = hashlib.sha256(username.encode('utf-8')).hexdigest()[:16]
        return f"anon_{hashed}"
    except ValueError:
        return username

def filter_and_transform(raw_data):
    """Mengambil hanya field yang diperlukan sesuai spesifikasi soal."""
    length_obj = raw_data.get('length') or {}
    return {
        'id': raw_data.get('id'),
        'type': raw_data.get('type'),
        'wiki': raw_data.get('wiki'),
        'server_name': raw_data.get('server_name'),
        'title': raw_data.get('title'),
        'namespace': raw_data.get('namespace'),
        'user': anonymize_user(raw_data.get('user')),
        'bot': bool(raw_data.get('bot', False)),
        'minor': bool(raw_data.get('minor', False)),
        'timestamp': raw_data.get('timestamp'),
        'length': {
            'old': length_obj.get('old'),
            'new': length_obj.get('new')
        },
        'comment': raw_data.get('comment', '')
    }

def delivery_report(err, msg):
    # Abaikan error timeout spesifik (-192) agar tidak spam log jika sedang retry internal
    if err is not None and err.code() != -192: 
        print(f"[ERROR] Gagal kirim pesan: {err}")

def run_producer():
    # Konfigurasi Producer Anti-Timeout & Kinerja Tinggi (Digabung dengan acks: all dari UTS)
    producer = Producer({
        'bootstrap.servers': KAFKA_BROKER,
        'acks': 'all',                             # Garansi pengiriman maksimal
        'compression.type': 'snappy',              # Kompresi ringan
        'linger.ms': 100,                          # Kumpulkan batch selama 100ms
        'message.timeout.ms': 120000,              # Timeout pesan 2 menit
        'request.timeout.ms': 60000,               # Timeout request 60 detik
        'queue.buffering.max.messages': 200000,    # Tampung hingga 200.000 pesan di RAM jika disk sibuk
        'delivery.report.only.error': True
    })

    msg_count = 0
    start_time = time.time()

    while True:
        try:
            print(f"[{time.strftime('%X')}] Menghubungkan ke Wikimedia EventStreams...")
            with requests.get(STREAM_URL, stream=True, headers=HEADERS, timeout=60) as response:
                for line in response.iter_lines(decode_unicode=True):
                    if line and line.startswith("data: "):
                        try:
                            raw_json = json.loads(line[6:])
                            if raw_json.get('meta', {}).get('domain') == 'canary':
                                continue
                            
                            # Fungsi dipanggil dengan aman karena sudah didefinisikan di atas
                            clean_payload = filter_and_transform(raw_json)
                            
                            producer.produce(
                                TOPIC_NAME,
                                value=json.dumps(clean_payload).encode('utf-8'),
                                callback=delivery_report
                            )
                            
                            producer.poll(0) # Memicu pengiriman tanpa memblokir proses
                            msg_count += 1

                            # Cetak log setiap 500 pesan sesuai format UTS
                            if msg_count % 500 == 0:
                                elapsed = time.time() - start_time
                                rate = msg_count / elapsed
                                print(
                                    f"[{time.strftime('%X')}] Terkirim: {msg_count} pesan | "
                                    f"Durasi: {elapsed:.1f}s | Laju: {rate:.2f} pesan/detik"
                                )
                                
                        except json.JSONDecodeError:
                            continue

        except Exception as e:
            print(f"[WARNING] Koneksi terputus ({e}). Menyambung ulang dalam 5 detik...")
            time.sleep(5)
        except KeyboardInterrupt:
            print("\n[INFO] Menghentikan Producer. Menyelesaikan pengiriman antrean (mohon tunggu)...")
            producer.flush(timeout=30) # Hanya flush saat user menekan Stop
            elapsed = time.time() - start_time
            rate = msg_count / elapsed if elapsed > 0 else 0
            print(f"[SELESAI] Producer dihentikan. Total: {msg_count} pesan | Laju: {rate:.2f} msg/s")
            break

# Jalankan fungsi
run_producer()

In [5]:
# CELL 4: Bukti Soal 1b - Cek Jumlah Pesan & Offset Tiap Partisi Kafka
# (Screenshot output cell ini untuk bukti laporan Soal 1b)
from confluent_kafka import Consumer, TopicPartition

BROKER = '192.168.110.109:9092'
TOPIC = 'wikipedia-recentchange'

c_meta = Consumer({'bootstrap.servers': BROKER, 'group.id': 'cek-offset-group'})
metadata = c_meta.list_topics(TOPIC, timeout=10)
partitions = [TopicPartition(TOPIC, p) for p in metadata.topics[TOPIC].partitions.keys()]

total_pesan = 0
print("=== BUKTI OFFSET TIAP PARTISI KAFKA ===")
for tp in sorted(partitions, key=lambda x: x.partition):
    low, high = c_meta.get_watermark_offsets(tp)
    jml = high - low
    total_pesan += jml
    print(f"Partisi {tp.partition} | Low Offset: {low} | High Offset: {high} | Jumlah: {jml}")

print(f"\nTOTAL PESAN TERKUMPUL: {total_pesan} pesan")
c_meta.close()

=== BUKTI OFFSET TIAP PARTISI KAFKA ===
Partisi 0 | Low Offset: 0 | High Offset: 36747 | Jumlah: 36747
Partisi 1 | Low Offset: 0 | High Offset: 43377 | Jumlah: 43377
Partisi 2 | Low Offset: 0 | High Offset: 39248 | Jumlah: 39248

TOTAL PESAN TERKUMPUL: 119372 pesan


In [6]:
# CELL 5: Bukti Soal 1c - Uji Decoupling (Toleransi Loading)
import json
from confluent_kafka import Consumer, TopicPartition

c_uji = Consumer({
    'bootstrap.servers': '192.168.110.109:9092',
    'group.id': 'test-decoupling-final',
    'enable.auto.commit': False,
    'auto.offset.reset': 'earliest'
})

# Langsung baca dari Partisi 0
tp0 = TopicPartition('wikipedia-recentchange', 0, 0)
c_uji.assign([tp0])

print("Membaca 50 pesan pertama dari Partisi 0 (Mohon tunggu)...")
dibaca = 0
percobaan_kosong = 0

while dibaca < 50:
    msg = c_uji.poll(timeout=5.0)
    
    if msg is None:
        percobaan_kosong += 1
        print(f"  -> Loading data (percobaan {percobaan_kosong}/5)...")
        if percobaan_kosong >= 5:
            print("[INFO] Kafka gagal merespons. Pastikan Kafka di Panel Lab menyala.")
            break
        continue  # Coba tarik data lagi, jangan langsung berhenti
        
    if msg.error():
        continue
        
    dibaca += 1
    if dibaca == 1 or dibaca == 50:
        data = json.loads(msg.value().decode('utf-8'))
        print(f"Batch 1 - Pesan ke-{dibaca} | Partisi: {msg.partition()} | Offset: {msg.offset()} | Wiki: {data.get('wiki')}")

# Simpan posisi berhenti di offset 50 jika berhasil
if dibaca == 50:
    c_uji.commit(offsets=[TopicPartition('wikipedia-recentchange', 0, 50)], asynchronous=False)
    print("\n[STOP] Consumer dimatikan pada Offset 50! Tunggu 2 menit, lalu jalankan Cell 6.")
    
c_uji.close()

Membaca 50 pesan pertama dari Partisi 0 (Mohon tunggu)...
Batch 1 - Pesan ke-1 | Partisi: 0 | Offset: 0 | Wiki: wikidatawiki
Batch 1 - Pesan ke-50 | Partisi: 0 | Offset: 49 | Wiki: enwiki

[STOP] Consumer dimatikan pada Offset 50! Tunggu 2 menit, lalu jalankan Cell 6.


In [8]:
# CELL 6: Lanjutan Uji Decoupling
import json
from confluent_kafka import Consumer, TopicPartition

c_lanjut = Consumer({
    'bootstrap.servers': '192.168.110.109:9092',
    'group.id': 'test-decoupling-final',
    'enable.auto.commit': False,
    'auto.offset.reset': 'earliest'
})

# Ambil offset ke-50 yang disimpan oleh Cell 5
committed = c_lanjut.committed([TopicPartition('wikipedia-recentchange', 0)], timeout=15)
last_offset = committed[0].offset
print(f"Melanjutkan pembacaan dari Offset tersimpan: {last_offset}")

# Lanjutkan pembacaan mulai dari offset 50
c_lanjut.assign([TopicPartition('wikipedia-recentchange', 0, last_offset)])

dibaca = 0
percobaan_kosong = 0
while dibaca < 10:
    msg = c_lanjut.poll(timeout=5.0)
    if msg is None:
        percobaan_kosong += 1
        print(f"  -> Menunggu respons broker ({percobaan_kosong}/8)...")
        if percobaan_kosong >= 8:
            print("[INFO] Broker masih timeout. Restart Kafka di Panel Lab lalu jalankan ulang Cell 6.")
            break
        continue
    if msg.error():
        continue
        
    dibaca += 1
    data = json.loads(msg.value().decode('utf-8'))
    print(f"Batch 2 - Lanjut ke-{dibaca} | Partisi: {msg.partition()} | Offset: {msg.offset()} | Wiki: {data.get('wiki')}")

c_lanjut.close()
if dibaca == 10:
    print("\n[SELESAI] Uji Decoupling Berhasil! Lanjut ke Cell 7 (Export Data).")

Melanjutkan pembacaan dari Offset tersimpan: 50
Batch 2 - Lanjut ke-1 | Partisi: 0 | Offset: 50 | Wiki: commonswiki
Batch 2 - Lanjut ke-2 | Partisi: 0 | Offset: 51 | Wiki: enwiktionary
Batch 2 - Lanjut ke-3 | Partisi: 0 | Offset: 52 | Wiki: commonswiki
Batch 2 - Lanjut ke-4 | Partisi: 0 | Offset: 53 | Wiki: wikidatawiki
Batch 2 - Lanjut ke-5 | Partisi: 0 | Offset: 54 | Wiki: commonswiki
Batch 2 - Lanjut ke-6 | Partisi: 0 | Offset: 55 | Wiki: commonswiki
Batch 2 - Lanjut ke-7 | Partisi: 0 | Offset: 56 | Wiki: enwiki
Batch 2 - Lanjut ke-8 | Partisi: 0 | Offset: 57 | Wiki: wikidatawiki
Batch 2 - Lanjut ke-9 | Partisi: 0 | Offset: 58 | Wiki: commonswiki
Batch 2 - Lanjut ke-10 | Partisi: 0 | Offset: 59 | Wiki: ttwiki

[SELESAI] Uji Decoupling Berhasil! Lanjut ke Cell 7 (Export Data).


In [9]:
# CELL 7: Ekspor Seluruh Data Kafka ke File Snapshot JSONL (Untuk Soal 2 - 4)
from confluent_kafka import Consumer, TopicPartition

c_export = Consumer({
    'bootstrap.servers': '192.168.110.109:9092',
    'group.id': 'export-snapshot-group',
    'enable.auto.commit': False
})

topic = 'wikipedia-recentchange'
partitions = [0, 1, 2]
target_offsets = {}
assign_list = []
total_target = 0

for p in partitions:
    low, high = c_export.get_watermark_offsets(TopicPartition(topic, p), timeout=15)
    target_offsets[p] = high
    jumlah_p = high - low
    total_target += jumlah_p
    if jumlah_p > 0:
        assign_list.append(TopicPartition(topic, p, low))
    print(f"Partisi {p}: Offset {low} s/d {high} ({jumlah_p} pesan)")

print(f"\nMengekspor total {total_target} pesan ke 'wikipedia_snapshot.jsonl'...")

c_export.assign(assign_list)
selesai_partisi = {p: (target_offsets[p] == 0) for p in partitions}
jumlah_terekspor = 0
kosong = 0

with open('wikipedia_snapshot.jsonl', 'w', encoding='utf-8') as f:
    while not all(selesai_partisi.values()):
        msg = c_export.poll(timeout=5.0)
        if msg is None:
            kosong += 1
            if kosong >= 10:
                print("[INFO] Timeout dari broker saat ekspor.")
                break
            continue
        if msg.error():
            continue
        
        kosong = 0
        p = msg.partition()
        off = msg.offset()
        
        if off < target_offsets[p]:
            f.write(msg.value().decode('utf-8') + '\n')
            jumlah_terekspor += 1
            if jumlah_terekspor % 25000 == 0:
                print(f"  -> Progres: {jumlah_terekspor} / {total_target} pesan...")
                
        if off + 1 >= target_offsets[p]:
            selesai_partisi[p] = True

c_export.close()
print(f"\n[SELESAI] Berhasil mengekspor {jumlah_terekspor} pesan ke file 'wikipedia_snapshot.jsonl'!")

Partisi 0: Offset 0 s/d 36747 (36747 pesan)
Partisi 1: Offset 0 s/d 43377 (43377 pesan)
Partisi 2: Offset 0 s/d 39248 (39248 pesan)

Mengekspor total 119372 pesan ke 'wikipedia_snapshot.jsonl'...
  -> Progres: 25000 / 119372 pesan...
  -> Progres: 50000 / 119372 pesan...
  -> Progres: 75000 / 119372 pesan...
  -> Progres: 100000 / 119372 pesan...

[SELESAI] Berhasil mengekspor 119372 pesan ke file 'wikipedia_snapshot.jsonl'!
